# Outline:

Constraint insertion: 
- Min/max order quantity (one quanity per price, mulitple entries possible per product)
- Min/max per quality attribute (ie absence_of_heavymetals)
- Min/max price

Error specifications and ranking:
- Deviation metrics (ie. KL)
- Filter out options of uncertainty (None of specified range)

## Get attibutes from text2product

In [ ]:
import importlib
import re
import sys
sys.path.insert(0, ".")

import db, text2product
importlib.reload(text2product)
importlib.reload(db)

from pathlib import Path
from text2product import html_to_text, extract_product_info
from db import upsert_supplier_product_prices, upsert_supplier_product_info, get_processed_supplier_products

SUPPLIER_PRODUCTS_DIR = Path("../data/supplier_products")
FNAME_RE = re.compile(r"^(\d+)_.+?_(\d+)_(RM-[^.]+)\.html$")
FORCE = False  # set True to reprocess everything from scratch

already_done = set() if FORCE else get_processed_supplier_products()
files = sorted(SUPPLIER_PRODUCTS_DIR.glob("*.html"))
print(f"Processing {len(files)} files ({len(already_done)} already done)...")

for path in files:
    m = FNAME_RE.match(path.name)
    if not m:
        print(f"  SKIP (unrecognised filename): {path.name}")
        continue

    supplier_id, product_id, sku_slug = int(m.group(1)), int(m.group(2)), m.group(3)

    if (supplier_id, product_id) in already_done:
        continue

    try:
        page_text = html_to_text(path.read_text(errors="replace"))
        result = extract_product_info(page_text, sku_slug)
    except Exception as e:
        print(f"  ERROR {path.name}: {e}")
        continue

    prices        = result.get("prices") or []
    purity        = result.get("purity")
    quality       = result.get("quality")
    quality_score = result.get("quality_score")
    compliance    = result.get("compliance") or {}

    if prices:
        upsert_supplier_product_prices(supplier_id, product_id, prices)
    upsert_supplier_product_info(supplier_id, product_id, purity, quality, quality_score, compliance)
    already_done.add((supplier_id, product_id))

    metrics_present = [k for k, v in compliance.items() if v is not None]
    print(f"  {path.name}: {len(prices)} tier(s), purity={purity}, quality={quality!r}, metrics={metrics_present}")

print("Done.")


Processing 203 files (1 already done)...
  24_Nutra_Blend_282_RM-C13-energy-support-botanicals-nutrients-4823efcb.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_582_RM-C33-ferment-media-c911f8c0.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_751_RM-C45-prebiotic-fiber-19375eea.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_820_RM-C52-performance-support-nutrients-165afec4.html: 1 tier(s), purity=0.84, quality='feed grade', metrics=['assay_potency']
  24_Nutra_Blend_885_RM-C57-cultured-nutrients-5c87fee4.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_886_RM-C57-fruit-nutrients-b65036d4.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_906_RM-C57-vegetable-nutrients-4667fd2d.html: 1 tier(s), purity=None, quality=None, metrics=[]
  24_Nutra_Blend_963_RM-C60-organic-food-complex-83c82fa3.html: 1 tier(s), purity=None, quality='feed

In [ ]:
# Export extracted data to JSON so others can import without re-running Gemini
import importlib, json, sys
sys.path.insert(0, ".")
import db; importlib.reload(db)
from db import get_supplier_products_enriched
from pathlib import Path

out = Path("../data/extracted_products.json")
data = get_supplier_products_enriched()
# only export rows that have been processed
data = [d for d in data if d.get("purity") is not None or d.get("quality") is not None or any(d.get("compliance", {}).values()) or d.get("prices")]
out.write_text(json.dumps(data, indent=2))
print(f"Exported {len(data)} processed rows to {out}")


In [ ]:
# Import extracted data into a fresh DB (run this instead of the Gemini extraction cell)
import importlib, json, sys
sys.path.insert(0, ".")
import db; importlib.reload(db)
from db import upsert_supplier_product_prices, upsert_supplier_product_info

data = json.loads(Path("../data/extracted_products.json").read_text())
for d in data:
    if d.get("prices"):
        upsert_supplier_product_prices(d["supplier_id"], d["product_id"], d["prices"])
    upsert_supplier_product_info(
        d["supplier_id"], d["product_id"],
        d.get("purity"), d.get("quality"), d.get("quality_score"), d.get("compliance"),
    )
print(f"Imported {len(data)} rows.")


## Constraint filtering

In [ ]:
import importlib
import sys
sys.path.insert(0, ".")

import db, text2product, filter_products
importlib.reload(text2product)
importlib.reload(filter_products)
importlib.reload(db)

from db import get_supplier_products_enriched
from filter_products import filter_products as apply_filters, make_filters

price_range    = (None, None)
quantity_range = (None, None)
purity_range   = (None, None)
quality_range  = (None, None)  # 0-1: 0.7=food/kosher, 0.9=GMP, 1.0=USP/pharma
quality_metrics = {
    # "identity_confidence": (0.95, None),
    # "assay_potency":       (0.97, 1.03),
    # "heavy_metals":        (None, 10.0),
    # "pesticide_residues":  (None, 100.0),
    # "microbial_limits":    (1.0,  None),
    # "moisture_content":    (None, 0.05),
    # "residual_solvents":   (None, 410.0),
}

products = get_supplier_products_enriched()
filters  = make_filters(price_range, quantity_range, purity_range, quality_range, quality_metrics)
results  = apply_filters(products, filters)

print(f"{len(results)}/{len(products)} products passed")
results


Example

In [ ]:
products = get_supplier_products_enriched()

filters = make_filters(
    price_range=(None, 50.0),
    quantity_range=(100.0, None),
    purity_range=(0.98, None),
    quality_range=(0.9, None),
    quality_metrics={
        "identity_confidence": (0.95, None),
        "assay_potency":       (0.97, 1.03),
        "heavy_metals":        (None, 10.0),
        "microbial_limits":    (1.0,  None),
    },
)

results = apply_filters(products, filters)
print(f"{len(results)}/{len(products)} products passed")
results


## Ranking function

In [ ]:
import importlib
import sys
sys.path.insert(0, ".")

import db, filter_products
importlib.reload(db)
importlib.reload(filter_products)

from db import batch, get_supplier_products_enriched, get_boms
from filter_products import filter_products as apply_filters, make_filters


def score_product(p: dict, weights: dict | None = None) -> float:
    """Composite score ∈ [0, 1]; returns 0.0 when no data is available."""
    w = weights or {
        "quality_score":       0.40,
        "purity":              0.30,
        "identity_confidence": 0.15,
        "assay_potency":       0.15,
    }
    score, total_w = 0.0, 0.0
    compliance = p.get("compliance") or {}

    for key, wt in w.items():
        if key == "quality_score":
            v = p.get("quality_score")
        elif key == "purity":
            v = p.get("purity")
        elif key == "assay_potency":
            v = compliance.get(key)
            if v is not None:
                v = 1.0 - abs(v - 1.0)   # penalise deviation from target 1.0
        else:
            v = compliance.get(key)

        if v is not None:
            score   += wt * min(max(v, 0.0), 1.0)
            total_w += wt

    return round(score / total_w, 4) if total_w > 0 else 0.0


def rank_suppliers(
    finished_good_sku: str,
    filters: list | None = None,
    weights: dict | None = None,
) -> dict:
    """
    Calls batch() to assign BOM components to the fewest suppliers, then scores
    each assignment by composite quality/compliance metrics.

    Returns:
        suppliers:   ordered list of chosen supplier names (from batch)
        assignments: {sku: supplier_name}
        uncovered:   SKUs with no available supplier
        ranked:      list of assignment dicts sorted by score descending
    """
    batch_result = batch(finished_good_sku)
    if not batch_result["assignments"]:
        return {**batch_result, "ranked": []}

    all_products = get_supplier_products_enriched()
    if filters:
        all_products = apply_filters(all_products, filters)

    enriched_idx = {(p["supplier_name"], p["sku"]): p for p in all_products}

    ranked = []
    for sku, supplier_name in batch_result["assignments"].items():
        p = enriched_idx.get((supplier_name, sku), {})
        ranked.append({
            "supplier":      supplier_name,
            "sku":           sku,
            "score":         score_product(p, weights),
            "purity":        p.get("purity"),
            "quality":       p.get("quality"),
            "quality_score": p.get("quality_score"),
            "compliance":    p.get("compliance") or {},
            "prices":        p.get("prices") or [],
        })

    ranked.sort(key=lambda x: x["score"], reverse=True)
    return {**batch_result, "ranked": ranked}


# ── Example: pick first available finished-good SKU from the DB ──────────────
boms = get_boms()
if not boms:
    print("No BOMs found in DB.")
else:
    example_sku = boms[0]["ProducedSKU"]
    result = rank_suppliers(example_sku)

    print(f"Finished good : {example_sku}")
    print(f"Suppliers chosen: {result['suppliers']}")
    print(f"Uncovered SKUs  : {result['uncovered']}")
    print()

    if result["ranked"]:
        top = result["ranked"][0]
        print(f"── Top assignment ──────────────────────────────")
        print(f"  Supplier  : {top['supplier']}")
        print(f"  SKU       : {top['sku']}")
        print(f"  Score     : {top['score']}")
        print(f"  Purity    : {top['purity']}")
        print(f"  Quality   : {top['quality']}  (quality_score={top['quality_score']})")
        print(f"  Compliance metrics:")
        for k, v in top["compliance"].items():
            if v is not None:
                print(f"    {k:25s}: {v}")
        print(f"  Price tiers:")
        for tier in top["prices"]:
            qty  = f"{tier['quantity']} {tier['unit']}" if tier.get("quantity") else "—"
            print(f"    {qty:20s} @ {tier['price']} {tier['currency']}")


## Evaluation

- make 10 queries with manual tagged correctness
- determne correctness comparing results of model top-5 suggestions